In [2]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/cleaned/kaggle_cleaned.csv")

print("SHAPE:")
print(df.shape)

print("\nCOLUMNS:")
print(df.columns.tolist())

print("\nDATATYPES:")
print(df.dtypes)

print("\nNULL COUNTS:")
print(df.isnull().sum())

print("\nSAMPLE:")
print(df.head(3))

SHAPE:
(97682, 17)

COLUMNS:
['job_title', 'job_id', 'currency', 'job_posted_date', 'company_name', 'skills', 'experience', 'salary', 'location', 'company_id', 'reviews_count', 'company_rating', 'job_description', 'minimum_salary', 'maximum_salary', 'minimum_experience', 'maximum_experience']

DATATYPES:
job_title              object
job_id                  int64
currency               object
job_posted_date        object
company_name           object
skills                 object
experience             object
salary                 object
location               object
company_id              int64
reviews_count         float64
company_rating        float64
job_description        object
minimum_salary        float64
maximum_salary        float64
minimum_experience    float64
maximum_experience    float64
dtype: object

NULL COUNTS:
job_title             0
job_id                0
currency              0
job_posted_date       0
company_name          0
skills                0
experience  

In [4]:
# Load files

naukri = pd.read_csv("../data/cleaned/naukri_final_dataset.csv")
foundit = pd.read_csv("../data/cleaned/foundit_final_dataset.csv")
timesjobs = pd.read_csv("../data/cleaned/timesjobs_final_dataset.csv")
serpapi = pd.read_csv("../data/cleaned/serpapi_jobs_clean.csv")
kaggle = pd.read_csv("../data/cleaned/kaggle_cleaned.csv")

In [5]:
# standardize column names

def clean_columns(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

datasets = [naukri, foundit, timesjobs, serpapi, kaggle]

for df in datasets:
    clean_columns(df)

In [6]:
# adding portal names

serpapi["portal_name"] = "google_jobs"
kaggle["portal_name"] = "kaggle_jobs"

In [7]:
# rename SerpAPI columns

serpapi = serpapi.rename(columns={
    "search_query": "role_search",
    "job_title": "job_title",
    "company_name": "company_name",
    "location": "location",
    "description": "job_description",
    "via": "source_platform",
    "job_id": "source_job_id"
})

In [8]:
# rename kaggle columns

kaggle = kaggle.rename(columns={
    "skills": "skills_required",
    "experience": "experience_required",
    "job_posted_date": "posted_date",
    "job_id": "source_job_id"
})

In [9]:
print(naukri.columns)
print(foundit.columns)
print(timesjobs.columns)
print(serpapi.columns)
print(kaggle.columns)

Index(['portal_name', 'role_search', 'job_title', 'company_name',
       'experience_required', 'salary', 'location', 'skills_required',
       'posted_date', 'job_description', 'company_rating', 'reviews_count',
       'job_type', 'company_size', 'industry', 'apply_link'],
      dtype='object')
Index(['portal_name', 'role_search', 'job_title', 'company_name',
       'experience_required', 'salary', 'location', 'skills_required',
       'posted_date', 'job_description', 'company_rating', 'reviews_count',
       'job_type', 'company_size', 'industry', 'apply_link'],
      dtype='object')
Index(['portal_name', 'role_search', 'job_title', 'company_name',
       'experience_required', 'salary', 'location', 'skills_required',
       'posted_date', 'job_description', 'company_rating', 'reviews_count',
       'job_type', 'company_size', 'industry', 'apply_link'],
      dtype='object')
Index(['role_search', 'job_title', 'company_name', 'location',
       'source_platform', 'job_description', '

In [10]:
# Master Schema

master_columns = [
    'portal_name',
    'role_search',
    'job_title',
    'company_name',
    'experience_required',
    'salary',
    'location',
    'skills_required',
    'posted_date',
    'job_description',
    'company_rating',
    'reviews_count',
    'job_type',
    'company_size',
    'industry',
    'apply_link',
    'source_platform',
    'source_job_id',
    'thumbnail',
    'job_highlights',
    'detected_extensions',
    'currency',
    'company_id',
    'minimum_salary',
    'maximum_salary',
    'minimum_experience',
    'maximum_experience'
]

In [11]:
# Add missing columns 

all_datasets = [naukri, foundit, timesjobs, serpapi, kaggle]

for df in all_datasets:
    for col in master_columns:
        if col not in df.columns:
            df[col] = np.nan

In [12]:
# Reorder columns

naukri = naukri[master_columns]
foundit = foundit[master_columns]
timesjobs = timesjobs[master_columns]
serpapi = serpapi[master_columns]
kaggle = kaggle[master_columns]

In [13]:
# Merge all data sets

master_jobs = pd.concat(
    [naukri, foundit, timesjobs, serpapi, kaggle],
    ignore_index=True
)

In [14]:
# Check final shape

master_jobs.shape

(106120, 27)

In [15]:
# Check duplicates

master_jobs.duplicated().sum()

np.int64(92)

In [17]:
# remove duplicates

master_jobs = master_jobs.drop_duplicates()

In [16]:
# Check null percentage

null_percent = (
    master_jobs.isnull().sum() / len(master_jobs)
) * 100

null_percent.sort_values(ascending=False)

industry               100.000000
company_size           100.000000
job_type               100.000000
job_highlights         100.000000
thumbnail               99.959480
source_platform         99.934037
detected_extensions     99.934037
apply_link              92.734640
role_search             92.048624
maximum_salary           7.951376
minimum_salary           7.951376
minimum_experience       7.951376
maximum_experience       7.951376
currency                 7.951376
company_id               7.951376
source_job_id            7.885413
salary                   6.791368
company_rating           2.316246
reviews_count            2.316246
skills_required          1.084621
experience_required      0.246890
location                 0.201658
posted_date              0.065963
job_description          0.052770
company_name             0.000000
portal_name              0.000000
job_title                0.000000
dtype: float64

In [18]:
# Dropping very high null columns

drop_cols = [
    'industry',
    'company_size',
    'job_type',
    'job_highlights',
    'thumbnail',
    'source_platform',
    'detected_extensions'
]

master_jobs = master_jobs.drop(columns=drop_cols)

In [19]:
master_jobs.shape

(106028, 20)

In [ ]:
# fill text columns

master_jobs["skills_required"] = master_jobs["skills_required"].fillna("Not Mentioned")

master_jobs["role_search"] = master_jobs["role_search"].fillna("Unknown")

master_jobs["location"] = master_jobs["location"].fillna("Unknown")

In [ ]:
# fill numeric rating columns

master_jobs["company_rating"] = master_jobs["company_rating"].fillna(0)

master_jobs["reviews_count"] = master_jobs["reviews_count"].fillna(0)

In [23]:
# exporting to csv

master_jobs.to_csv(
    "../data/cleaned/master_jobs_dataset.csv",
    index=False
)